# Tech Challenge - Data Engineering

## 02 - Transformação | Camada Silver

### Objetivo

Este notebook é responsável pelo tratamento, padronização, validação e integração dos dados provenientes da camada Bronze.

A camada Silver deverá preservar a granularidade necessária para análise, corrigindo problemas de qualidade e preparando os dados para a construção dos indicadores da camada Gold.

### Fonte

As tabelas utilizadas neste notebook são provenientes do schema `bronze`, criado durante a etapa de "ingestão bronze"

# Configuração

## Bibliotecas

In [0]:
from pyspark.sql import functions as F

# Conhecendo os Dados

## Carregando Tabelas

In [0]:
tabelas = [
    "alunos",
    "municipio",
    "uf",
    "meta_alfabetizacao_municipio",
    "meta_alfabetizacao_uf",
    "meta_alfabetizacao_brasil",
    "dicionario",
]

bronze = {tabela: spark.table(f"bronze.{tabela}") for tabela in tabelas}

## Inspeção

Schemas e amostra dos dados

In [0]:
for tabela, df in bronze.items():
    print("=" * 80)
    print(f"TABELA: bronze.{tabela}")
    print(f"Colunas: {len(df.columns)}")

    df.printSchema()
    df.show(10, truncate=False)

## Diagnóstico de duplicidades

Antes da aplicação de qualquer regra de deduplicação, são avaliadas as chaves candidatas de cada tabela.

A deduplicação somente será realizada caso sejam encontrados registros repetidos e exista uma regra clara para determinar qual registro deve ser preservado.

In [0]:
chaves_candidatas = {
    "alunos": ["ano", "id_aluno"],
    "municipio": ["ano", "id_municipio", "serie", "rede"],
    "uf": ["ano", "sigla_uf", "serie", "rede"],
    "meta_alfabetizacao_municipio": ["ano", "id_municipio", "rede"],
    "meta_alfabetizacao_uf": ["ano", "sigla_uf", "rede"],
    "meta_alfabetizacao_brasil": ["ano", "rede"],
    "dicionario": ["id_tabela", "nome_coluna", "chave", "cobertura_temporal"],
}

In [0]:
resultado_duplicidades = []

for tabela, chaves in chaves_candidatas.items():

    df = bronze[tabela]

    duplicados = df.groupBy(*chaves).count().filter(F.col("count") > 1)

    qtd_grupos_duplicados = duplicados.count()

    resultado_duplicidades.append((tabela, ", ".join(chaves), qtd_grupos_duplicados))

df_duplicidades = spark.createDataFrame(
    resultado_duplicidades, ["tabela", "chave_candidata", "grupos_duplicados"]
)

display(df_duplicidades)

## Diagnóstico de valores nulos

Nesta etapa é avaliada a completude das tabelas da camada Bronze.

A presença de valores nulos não implica necessariamente um problema de qualidade.

Alguns campos podem não estar disponíveis em determinados anos ou podem ser nulos em função da própria regra do dado, como no caso de alunos ausentes que não possuem proficiência registrada.

O objetivo desta análise é identificar esses padrões antes da definição das regras da camada Silver.

In [0]:
def perfil_nulos(df, tabela):

    total = df.count()

    expressoes = []

    for coluna in df.columns:

        # Não precisamos analisar os metadados técnicos
        if coluna.startswith("_"):
            continue

        expressoes.append(
            F.sum(F.when(F.col(coluna).isNull(), 1).otherwise(0)).alias(coluna)
        )

    resultado = df.agg(*expressoes).collect()[0]

    linhas = []

    for coluna in resultado.__fields__:

        qtd_nulos = resultado[coluna]

        percentual = qtd_nulos / total * 100 if total > 0 else 0

        linhas.append((tabela, coluna, total, qtd_nulos, round(percentual, 2)))

    return linhas

In [0]:
resultado_nulos = []

for tabela, df in bronze.items():
    resultado_nulos.extend(perfil_nulos(df, tabela))

df_nulos = spark.createDataFrame(
    resultado_nulos,
    ["tabela", "coluna", "total_linhas", "qtd_nulos", "percentual_nulos"],
)

display(df_nulos.orderBy(F.desc("percentual_nulos")))

In [0]:
(
    bronze["alunos"]
    .groupBy(
        "presenca",
        "preenchimento_caderno"
    )
    .agg(
        F.count("*").alias("qtd_alunos"),

        F.sum(
            F.when(
                F.col("proficiencia").isNull(),
                1
            ).otherwise(0)
        ).alias("proficiencia_nula"),

        F.sum(
            F.when(
                F.col("peso_aluno").isNull(),
                1
            ).otherwise(0)
        ).alias("peso_nulo")
    )
    .orderBy(
        "presenca",
        "preenchimento_caderno"
    )
    .show()
)

# Construção da Camada Silver

Após a análise exploratória e o diagnóstico de qualidade da camada Bronze, foram definidas as transformações necessárias para a camada Silver.

Os principais critérios adotados foram:

* Juntar tabelas que transmitem a mesma informação, como as de resultado territorial;
* Agregar a descrição dos campos nas tabelas, a partir do dicionário
* Criação de uma tabela para a distribuição dos níveis de proficiência
* Consolidação das metas em uma tabela única
* Não há valores duplicados, e os nulos representam ausências legítimas da informação, como do caso da proporção de alunos por nível, que passou a ser coletada apenas em 2024

A camada Silver será composta por quatro tabelas:

`silver.alunos`  
`silver.resultados_territoriais`  
`silver.distribuicao_niveis`  
`silver.metas`

Criar o schema

In [0]:
silver_schema = "silver"

spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS {silver_schema}"
)

print(f"Schema '{silver_schema}' disponível.")

## 01. Silver - Alunos

A tabela de alunos mantém a granularidade original por estudante.

Os códigos existentes na fonte são preservados e enriquecidos com suas respectivas descrições, utilizando o dicionário oficial disponibilizado pela Base dos Dados.

São enriquecidos os campos:

- série;
- rede;
- presença;
- preenchimento do caderno;
- indicador de alfabetização.

Os campos `proficiencia` e `peso_aluno` permanecem nulos quando a prova não foi preenchida, pois é a estrutura da base

### Função de Add Descrição

In [0]:
df_dicionario = bronze["dicionario"]


def adicionar_descricao_codigo(df, id_tabela, nome_coluna, nome_descricao):
    """
    Adiciona a descrição de uma coluna codificada utilizando
    a tabela de dicionário da Base dos Dados.
    """

    chave_auxiliar = f"__chave_{nome_coluna}"

    mapa = df_dicionario.filter(
        (F.col("id_tabela") == id_tabela) & (F.col("nome_coluna") == nome_coluna)
    ).select(F.col("chave").alias(chave_auxiliar), F.col("valor").alias(nome_descricao))

    return df.join(
        F.broadcast(mapa),
        F.col(nome_coluna).cast("string") == F.col(chave_auxiliar),
        "left",
    ).drop(chave_auxiliar)

### Manipulações

In [0]:
df_alunos_silver = bronze["alunos"]

mapeamentos_alunos = {
    "serie": "serie_descricao",
    "rede": "rede_descricao",
    "presenca": "presenca_descricao",
    "preenchimento_caderno": "preenchimento_caderno_descricao",
    "alfabetizado": "alfabetizado_descricao",
}

for coluna, descricao in mapeamentos_alunos.items():

    df_alunos_silver = adicionar_descricao_codigo(
        df=df_alunos_silver,
        id_tabela="alunos",
        nome_coluna=coluna,
        nome_descricao=descricao,
    )


# Ordenação das colunas para facilitar a leitura
df_alunos_silver = df_alunos_silver.select(
    "ano",
    "id_municipio",
    "id_escola",
    "id_aluno",
    "caderno",
    "serie",
    "serie_descricao",
    "rede",
    "rede_descricao",
    "presenca",
    "presenca_descricao",
    "preenchimento_caderno",
    "preenchimento_caderno_descricao",
    "alfabetizado",
    "alfabetizado_descricao",
    "proficiencia",
    "peso_aluno",
    "_ingestion_timestamp",
    "_source_system",
    "_source_table",
).withColumn("_silver_processed_at", F.current_timestamp())


# Persistindo no formato delta
(
    df_alunos_silver.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.alunos")
)

### Validação
Comparando a quantidade de linhas da tabela bronze e da silver

In [0]:
print(
    f"Bronze: {bronze['alunos'].count():,}"
)

print(
    f"Silver: {spark.table('silver.alunos').count():,}"
)

display(
    spark.table("silver.alunos")
    .limit(20)
)

## 02. Silver - Resultados Territoriais

As tabelas de município e UF apresentam a mesma estrutura conceitual de indicadores

Para padronizar o modelo, ambas são consolidadas em uma única tabela, denominada: `resultados_territoriais`.

São criados dois campos:

- `nivel_geografico`: identifica se o registro representa MUNICIPIO ou UF;
- `id_geografia`: contém o código do município ou a sigla da UF.

As distribuições por nível de proficiência são retiradas desta tabela e tratadas separadamente em `silver.distribuicao_niveis`.

### Manipulações e Union

In [0]:
########## MUNICÍPIO ##########

# Carregando a tabela bronze
df_municipio = bronze["municipio"]

# Adicionando a descrição pra Série
df_municipio = adicionar_descricao_codigo(
    df_municipio, "municipio", "serie", "serie_descricao"
)

# Adicionando Descrição pra Rede
df_municipio = adicionar_descricao_codigo(
    df_municipio, "municipio", "rede", "rede_descricao"
)

#  Criando a coluna de "Nível Geográfico", e padronizando a tabela pro union posterior
df_resultado_municipio = df_municipio.select(
    "ano",
    F.lit("MUNICIPIO").alias("nivel_geografico"),
    F.col("id_municipio").alias("id_geografia"),
    "serie",
    "serie_descricao",
    "rede",
    "rede_descricao",
    "taxa_alfabetizacao",
    "media_portugues",
    "_ingestion_timestamp",
    "_source_system",
    "_source_table",
)


########## UF ##########

df_uf = bronze["uf"]

# Adicionando a descrição pra Série
df_uf = adicionar_descricao_codigo(df_uf, "uf", "serie", "serie_descricao")

# Adicionando Descrição pra Rede
df_uf = adicionar_descricao_codigo(df_uf, "uf", "rede", "rede_descricao")

#  Criando a coluna de "Nível Geográfico", e padronizando a tabela pro union posterior
df_resultado_uf = df_uf.select(
    "ano",
    F.lit("UF").alias("nivel_geografico"),
    F.col("sigla_uf").alias("id_geografia"),
    "serie",
    "serie_descricao",
    "rede",
    "rede_descricao",
    "taxa_alfabetizacao",
    "media_portugues",
    "_ingestion_timestamp",
    "_source_system",
    "_source_table",
)


# Unindo as duas tabelas padronizadas
df_resultados_territoriais = df_resultado_municipio.unionByName(
    df_resultado_uf
).withColumn("_silver_processed_at", F.current_timestamp())

# Persistindo a tabela na camada Silver
(
    df_resultados_territoriais.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.resultados_territoriais")
)

### Validação

In [0]:
qtd_origem = (
    bronze["municipio"].count()
    + bronze["uf"].count()
)

qtd_silver = (
    spark.table(
        "silver.resultados_territoriais"
    ).count()
)

print(f"Origem: {qtd_origem:,}")
print(f"Silver: {qtd_silver:,}")

## 03. Silver - Distribuição por Nível de Proficiência

As tabelas de Município e UF possuem nove colunas representando a proporção de estudantes nos níveis de proficiência de 0 a 8.

Na fonte original, essas informações estão em colunas:

`proporcao_aluno_nivel_0` até `proporcao_aluno_nivel_8`.

Para facilitar análises e agregações, essas colunas são pivotadas, criando:

- `nivel_proficiencia`;
- `proporcao_alunos`.

O diagnóstico da camada Bronze demonstrou que essas informações não estão disponíveis em 2023. Valores ausentes não são convertidos para zero, pois zero representaria uma proporção real de 0%, enquanto NULL representa informação não disponível.

### Função de Pivotagem

In [0]:
def transformar_niveis(df, nivel_geografico, coluna_geografia):

    expressao_niveis = """
        stack(
            9,
            0, proporcao_aluno_nivel_0,
            1, proporcao_aluno_nivel_1,
            2, proporcao_aluno_nivel_2,
            3, proporcao_aluno_nivel_3,
            4, proporcao_aluno_nivel_4,
            5, proporcao_aluno_nivel_5,
            6, proporcao_aluno_nivel_6,
            7, proporcao_aluno_nivel_7,
            8, proporcao_aluno_nivel_8
        ) as (
            nivel_proficiencia,
            proporcao_alunos
        )
    """

    return (
        df.select(
            "ano",
            F.lit(nivel_geografico).alias("nivel_geografico"),
            F.col(coluna_geografia).alias("id_geografia"),
            "serie",
            "serie_descricao",
            "rede",
            "rede_descricao",
            F.expr(expressao_niveis),
            "_ingestion_timestamp",
            "_source_system",
            "_source_table",
        )
        # Mantemos apenas distribuições efetivamente disponíveis.
        .filter(F.col("proporcao_alunos").isNotNull())
    )

### Manipulações e Union

In [0]:
# Aplicando na base de nível Municipal
df_niveis_municipio = transformar_niveis(df_municipio, "MUNICIPIO", "id_municipio")

# Aplicando na base de nível Estadual
df_niveis_uf = transformar_niveis(df_uf, "UF", "sigla_uf")

# Unindo as tabelas
df_distribuicao_niveis = df_niveis_municipio.unionByName(df_niveis_uf).withColumn(
    "_silver_processed_at", F.current_timestamp()
)

# Persistindo em formato delta
(
    df_distribuicao_niveis.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.distribuicao_niveis")
)

### Validação

In [0]:
# Checando se a distribuição por ano, nível_geográfico resulta no total 1, que seria o correto
display(
    spark.table("silver.distribuicao_niveis")
    .groupBy("ano", "nivel_geografico", "id_geografia", "rede")
    .agg(F.sum("proporcao_alunos").alias("proporcao_alunos"))
    .groupBy("ano", "nivel_geografico", "rede")
    .agg(F.min("proporcao_alunos").alias("menor"),
         F.max("proporcao_alunos").alias("maior"))
)

## 04. Silver - Metas de Alfabetização

As metas de alfabetização são disponibilizadas originalmente em três tabelas distintas:

- Município;
- UF;
- Brasil.

Além disso, as metas de 2024 a 2030 estão distribuídas em diferentes colunas.

A camada Silver padroniza as três granularidades geográficas e transforma as metas para o formato long.

IMPORTANTE:

- `ano_referencia`: ano da observação publicada pela fonte;
- `ano_meta`: ano ao qual a meta de alfabetização se refere.

Essa diferenciação preserva possíveis revisões das metas ao longo do tempo.

### Função Pivot e Padronização

In [0]:
def transformar_metas(df, nivel_geografico, coluna_geografia=None):

    expressao_metas = """
        stack(
            7,
            2024, meta_alfabetizacao_2024,
            2025, meta_alfabetizacao_2025,
            2026, meta_alfabetizacao_2026,
            2027, meta_alfabetizacao_2027,
            2028, meta_alfabetizacao_2028,
            2029, meta_alfabetizacao_2029,
            2030, meta_alfabetizacao_2030
        ) as (
            ano_meta,
            meta_alfabetizacao
        )
    """

    if coluna_geografia is None:
        id_geografia = F.lit("BRASIL")
    else:
        id_geografia = F.col(coluna_geografia)

    # Apenas a tabela municipal possui nivel_alfabetizacao
    if "nivel_alfabetizacao" in df.columns:
        nivel_alfabetizacao = F.col("nivel_alfabetizacao")
    else:
        nivel_alfabetizacao = F.lit(None).cast("long")

    return df.select(
        F.col("ano").alias("ano_referencia"),
        F.lit(nivel_geografico).alias("nivel_geografico"),
        id_geografia.alias("id_geografia"),
        "rede",
        F.col("taxa_alfabetizacao").alias("taxa_alfabetizacao_referencia"),
        nivel_alfabetizacao.alias("nivel_alfabetizacao_referencia"),
        F.col("percentual_participacao").alias("percentual_participacao_referencia"),
        F.expr(expressao_metas),
        "_ingestion_timestamp",
        "_source_system",
        "_source_table",
    )

### Manipulações e Union

In [0]:
# Aplicando a transformação na base de nível municipal
df_metas_municipio = transformar_metas(
    bronze["meta_alfabetizacao_municipio"],
    "MUNICIPIO",
    "id_municipio"
)

# Aplicando a transformação na base de nível estadual
df_metas_uf = transformar_metas(
    bronze["meta_alfabetizacao_uf"],
    "UF",
    "sigla_uf"
)

# Aplicando a transformação na base de nível nacional
df_metas_brasil = transformar_metas(
    bronze["meta_alfabetizacao_brasil"],
    "BRASIL"
)

# Unindo as 3 tabelas
df_metas_silver = (
    df_metas_municipio
    .unionByName(df_metas_uf)
    .unionByName(df_metas_brasil)
    .withColumn(
        "_silver_processed_at",
        F.current_timestamp()
    )
)

# Persistindo no formato delta
(
    df_metas_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.metas")
)

### Validação

In [0]:
display(
    spark.table("silver.metas")
    .orderBy(
        "nivel_geografico",
        "id_geografia",
        "ano_referencia",
        "ano_meta"
    )
    .limit(100)
)

Números a seguir validados com os indicadores públicos, consultados em pesquisa no Google

In [0]:
%sql
select * from silver.metas where ano_referencia = 2025 and ano_meta = 2025 and nivel_geografico = 'UF'

# Resultados Pipeline Silver

In [0]:
tabelas_silver = [
    "alunos",
    "resultados_territoriais",
    "distribuicao_niveis",
    "metas"
]

resumo_silver = []

for tabela in tabelas_silver:

    df = spark.table(
        f"silver.{tabela}"
    )

    resumo_silver.append(
        (
            tabela,
            df.count(),
            len(df.columns)
        )
    )

df_resumo_silver = spark.createDataFrame(
    resumo_silver,
    [
        "tabela",
        "quantidade_linhas",
        "quantidade_colunas"
    ]
)

display(df_resumo_silver)